<p class="lead">Nearly every modelling project in the laboratory — a QSAR model of a kinase target, a classifier of screening hits, a predictor of a clinical outcome from gingival-tissue transcriptomes — passes through the same seven decisions, in the same order, and almost every published failure happens in the same two of them. This unit is about the <em>workflow</em>, not the algorithms: trees, forests and networks are taught in the sister artificial-intelligence course. Here you learn how to lay out a supervised project so that the number you finally report is one you can defend in front of a referee — how to frame the question, how to wrap every preprocessing step inside a resampling scheme that fits it on the training fold only, how to compare against baselines that are embarrassingly simple, how to read a learning curve, how to look your own errors in the face, and how to recognise the many faces of <strong>leakage</strong>. We finish by building the same model twice — once naively, once correctly — and measuring, in \(R^2\) units, how much the naive version lied.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad enseña el flujo de trabajo del aprendizaje supervisado tal como se practica en el laboratorio: cómo se plantea la pregunta (regresión sobre pActivity, clasificación con un umbral o un problema de ordenamiento), cuál es la unidad de predicción cuando un compuesto tiene varias medidas, cómo se construyen cadenas de preprocesamiento con <code>caret</code> y <code>recipes</code> para que todo ajuste se realice únicamente con los datos de entrenamiento, por qué siempre se parte de modelos de referencia (la media, el vecino más cercano, un modelo lineal regularizado), cómo se leen las curvas de aprendizaje, cómo se analizan los errores compuesto por compuesto, y cuáles son las fuentes de fuga de información (<em>leakage</em>). El experimento central construye el mismo modelo de forma ingenua y de forma correcta y mide el optimismo del primero: en estos datos la selección de variables realizada fuera de la validación cruzada infla el \(R^2\) en unas 0,24 unidades con 150 compuestos y produce un \(R^2\) de +0,56 sobre ruido puro.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Lay out a supervised project end-to-end: question, target variable, unit of prediction, split, preprocessing, model, evaluation, error analysis.</li>
<li>Decide whether a question is really a regression, a classification at a threshold, or a ranking problem, and state what the unit of prediction is.</li>
<li>Build <code>caret</code> and <code>recipes</code> preprocessing chains — including a hand-written transformer — so that every preprocessing step is fitted inside the training fold only.</li>
<li>Always start from baselines (mean, nearest neighbour, a regularised linear model) and report improvement over them rather than a bare score.</li>
<li>Read a learning curve and decide whether the next effort should buy more data or more model capacity.</li>
<li>Perform error analysis: identify which compounds are predicted worst and explain why in chemical terms.</li>
<li>Recognise the five standard leakage sources and quantify the optimism each one produces.</li>
<li>Draft the model card that accompanies a model when it leaves your laptop.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read it once linearly, running each chunk as you reach it and reading its output before you move on; then re-run chunks while changing one thing at a time — the activity threshold, the feature family, the baseline — and watch the numbers move. Boxes marked <strong>Domain context</strong> define the chemistry and biology vocabulary in plain language; boxes marked <strong>Bridge</strong> point to the unit of this or a sister course that takes the idea further. This is the <strong>R edition</strong>: the same data, the same experiments and the same conclusions as the <a href="../../Python/U11_Supervised_Workflow/index.html">Python unit</a>, written with base R and <code>stats</code>, <code>data.table</code>, <code>ggplot2</code><sup class="c"><a href="#r11">[11]</a></sup>, <code>caret</code><sup class="c"><a href="#r1">[1]</a></sup>, <code>glmnet</code><sup class="c"><a href="#r21">[21]</a></sup> and <code>ranger</code><sup class="c"><a href="#r19">[19]</a></sup>. In a fresh Colab R runtime run once, in a cell of its own: <code>install.packages(c("data.table","caret","glmnet","ranger","patchwork","R.utils"))</code>. This unit assumes Units 4–6 (tidy data, exploratory analysis, curation) and Unit 9 (linear models); Unit 12 takes the single honest number produced here and asks how much it should be trusted.</p></div>



In [ ]:
## Setup: libraries, global seed, plotting defaults, data loader.
library(data.table); library(Matrix); library(ggplot2); library(patchwork)
library(glmnet);     library(ranger); library(caret);   library(recipes)
set.seed(0)
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10),
                panel.grid.minor = element_blank()))
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")

## The course data loader: ../data/<name> when knitting from the unit folder.
data_path <- function(name) file.path("..", "data", name)
load_data  <- function(name, ...) data.table::fread(data_path(name), ...)

cat("R", as.character(getRversion()), "| data.table", as.character(packageVersion("data.table")),
    "| caret", as.character(packageVersion("caret")), "| glmnet", as.character(packageVersion("glmnet")),
    "| ranger", as.character(packageVersion("ranger")), "| ggplot2", as.character(packageVersion("ggplot2")), "\n")



The printed versions are the ones every number below was produced with; quoting them is the cheapest form of reproducibility (Unit 2).

## 1. The seven decisions of a supervised project

A supervised project is a chain of seven decisions, of which only the fifth is about an algorithm — the least consequential of them.

1. **The question.** What will somebody *do* with the prediction — buy a compound, synthesise an analogue, send a patient for a biopsy? This fixes everything downstream, including which errors are expensive.
2. **The target.** Which measured quantity encodes the answer, on which scale, with which measurement error, and which records are usable at all.
3. **The unit of prediction.** One row = one *what*? One measurement, compound, patient, tissue site? Rows must be the things you will be asked to predict.
4. **The split.** Which data are hidden from the model, and whether hiding them that way mimics how the model will be used. Unit 12 is entirely about this decision.
5. **The preprocessing and the model.** Scaling, imputation, encoding, featurisation, then an estimator — all fitted on a training fold and applied, never refitted, to a test fold.
6. **The evaluation.** A metric matching the question, computed on data the chain never saw, reported with an uncertainty and against a baseline.
7. **The error analysis.** Which rows are wrong, whether they are wrong for a reason, and what that implies about the applicability of the model.

Most published disasters in machine learning for science happen in decisions 3, 4 and 5 — not in the choice of algorithm but in the mechanics of how the preprocessing was fitted. The survey of Kapoor and Narayanan found leakage-induced over-optimism in hundreds of papers across seventeen fields<sup class="c"><a href="#r3">[3]</a></sup>, and reviews of machine learning in genomics<sup class="c"><a href="#r6">[6]</a></sup> and chemistry<sup class="c"><a href="#r7">[7]</a></sup> make the same diagnosis. This unit makes those mistakes mechanically difficult in your own code.

<div class="box bridge"><h4>Domain context — ROCK1, IC50 and the pActivity scale</h4>
<p><strong>ROCK1</strong> (Rho-associated protein kinase 1) is an enzyme — a <em>kinase</em>, that is, a protein that attaches a phosphate group to other proteins and so switches them on or off. It sits downstream of the signalling protein RhoA and controls how a cell contracts its internal skeleton; drugs that block it (fasudil, ripasudil, belumosudil) are studied for glaucoma, fibrosis and cardiovascular disease, and it is one of this laboratory's own targets. A chemist measures how well a candidate blocks it and reports a concentration: the <strong>IC50</strong> halves the enzyme's activity, while <strong>Ki</strong> and <strong>Kd</strong> are related binding constants. A <em>smaller</em> concentration means a <em>better</em> compound, and the values span nine orders of magnitude, so the field works with <strong>pActivity</strong> = \(-\log_{10}\) (concentration in molar). pActivity 9 is an excellent inhibitor (1 nM), pActivity 5 is a weak one (10 µM), and one unit of pActivity is a tenfold change in potency. Because different laboratories and assay formats disagree by roughly half a log unit even for the same compound<sup class="c"><a href="#r13">[13]</a></sup><sup class="c"><a href="#r14">[14]</a></sup>, a model that predicts pActivity to ±0.5 is already at the level of the experiment itself. Our data are a frozen snapshot of the public database <strong>ChEMBL</strong><sup class="c"><a href="#r9">[9]</a></sup>, curated in Unit 6.</p></div>

The chunk below loads the curated table and summarises the columns that matter for this unit. Everything in Units 11–14 runs on this one file.



In [ ]:
## The curated compound-level table: one row per standardised structure (see data/README_rock1.md).
df <- load_data("rock1_chembl_curated.csv")
cat("curated table:", nrow(df), "compounds x", ncol(df), "columns\n")
cat("censored (all measurements are '>' or '<'):", sum(df$censored),
    "| with at least one exact value:", sum(!df$censored), "\n")
cols <- c("pActivity", "n_measurements", "pActivity_range", "MolWt", "MolLogP", "TPSA", "QED")
summ <- df[censored == FALSE, lapply(.SD, function(v)
             c(mean = mean(v), sd = sd(v), min = min(v), median = median(v), max = max(v))),
           .SDcols = cols]
summ <- as.data.frame(t(round(as.matrix(summ), 2)))
names(summ) <- c("mean", "sd", "min", "median", "max")
knitr::kable(summ, caption = "Summary of the modelling subset (compounds with at least one exact measurement).")



<p class="caption"><b>Table (Section 1).</b> Summary of the modelling subset. The 2,764 compounds with at least one exact measurement have a mean pActivity of 6.51 and a standard deviation of 1.11 log units, spanning 3.57 to 9.82 — four orders of magnitude of potency. Molecular weight averages 381.6 g/mol and MolLogP 3.52: drug-sized kinase inhibitors, not a random library. That standard deviation of 1.11 is the number every model here must beat, since predicting the mean already achieves it.</p>

### 1.1 Decision 2 — the target, and who is allowed into the table

The target is `pActivity`. Two groups of records require a decision before any modelling.

**Censored measurements.** When an assay reports "IC50 > 10 µM" the true value is unknown; all we know is that it lies above a bound. The curated table keeps those 1,516 compounds (`censored == TRUE`) rather than deleting them silently, because they are exactly the inactive compounds and discarding them makes a dataset look easier and more potent than reality. They cannot enter an ordinary regression as if the bound were the value, so this unit models the 2,764 compounds with at least one exact measurement and *states* that restriction; Unit 17 handles censoring properly.

**Replicate spread.** A compound measured four times in four laboratories has four answers. The table stores the median of the exact measurements plus the spread, flagging the compounds that disagree by more than one log unit. That spread is the experimental noise floor: no model can be more accurate than the data are reproducible<sup class="c"><a href="#r13">[13]</a></sup>.



In [ ]:
## The target distribution, exact vs censored, and the noise floor implied by replicates.
reg <- df[censored == FALSE]            ## modelling subset: exact values only
y   <- reg$pActivity
dd  <- data.table(pActivity = df$pActivity,
                  kind = ifelse(df$censored, "censored", "exact"))
p1 <- ggplot(dd, aes(pActivity, fill = kind)) +
  geom_histogram(bins = 40, alpha = 0.65, position = "identity") +
  geom_vline(xintercept = mean(y), linetype = "dashed", linewidth = 0.4) +
  scale_fill_manual(values = c(censored = "#d2691e", exact = "#2b5d9b")) +
  labs(x = "pActivity  (-log10 M)", y = "compounds", fill = NULL,
       title = "Target distribution") + theme(legend.position = "top")
rep_range <- reg[n_measurements > 1, pActivity_range]
p2 <- ggplot(data.frame(r = rep_range), aes(r)) +
  geom_histogram(bins = 30, fill = PAL["green"], alpha = 0.85) +
  geom_vline(xintercept = 1, linetype = "dashed", linewidth = 0.4) +
  labs(x = "max - min pActivity across replicates", y = "compounds",
       title = paste0("Replicate spread (n = ", length(rep_range), " compounds)"))
print(p1 | p2)
cat("target sd =", round(sd(y), 3),
    "| median replicate range =", round(median(rep_range), 3),
    "| compounds disagreeing by > 1 log unit:", sum(rep_range > 1), "\n")



<p class="caption"><b>Figure 1.</b> Left: the two populations of the curated table. The exact measurements (blue, 2,764 compounds) are roughly bell-shaped around pActivity 6.5, while the censored records (orange, 1,516 compounds) pile up at low potency — compounds somebody tested and found inactive, reported as "greater than" a screening concentration. Right: for the 375 compounds of the modelling subset measured more than once, the spread between the largest and smallest replicate. The median spread is zero — most of them were measured twice in agreement — but the tail reaches beyond one log unit for 55 compounds, and that tail is the empirical noise floor of the target. The dashed vertical line in the left panel is the mean of the exact values, 6.51; its standard deviation, 1.109, is the error of the laziest possible model.</p>

### 1.2 Decision 3 — the unit of prediction

The raw ChEMBL download has 5,089 rows, one per *measurement*. The curated table has 4,280 rows, one per *compound*. Which should be the unit of prediction?

If a row were a measurement, the same molecule could appear in both training and test set — ten times in training, twice in test — and the test score would partly measure the model's ability to remember a structure it has already seen. That is the simplest form of leakage, and it is why Unit 6 aggregated measurements to one row per standardised parent structure. The unit of prediction here is **one compound**, because the question — "should we buy or synthesise this molecule?" — is asked once per molecule.

The analogous decision in the clinical dataset of Unit 21 is sharper: the periodontitis series has two tissue sites per patient, so a row is a *site* but the independent unit is a *patient*. Whenever rows are nested inside a larger unit (measurements in compounds, sites in patients, frames in a trajectory) the unit of prediction is the outer one, and the split must respect it — the grouped cross-validation of Unit 12.

<div class="box warn"><h4>Common misconception</h4>
<p>"More rows is more data, so I should model measurements, not compounds." More rows that are copies of each other is not more information; it is the same information with a louder voice. It inflates the apparent sample size, breaks the independence assumption of every confidence interval you will later compute, and — if the duplicates straddle the split — inflates the test score as well.</p></div>

### 1.3 Decision 1 revisited — regression, classification, or ranking?

The same data support three different questions, and the choice is made by the use, not by the data.

- **Regression on pActivity.** Predict the number. Appropriate when the downstream decision uses the magnitude: "is this analogue likely to reach single-digit nanomolar?" Metrics: RMSE, MAE, \(R^2\), Spearman \(\rho\).
- **Classification at a threshold.** Predict active/inactive, where "active" means pActivity above some cut-off. Appropriate when the decision is binary and the threshold is a real decision boundary (a screening cut-off, a potency gate for a project). Metrics: ROC-AUC, precision/recall, MCC. The cost: a compound at 6.99 and one at 7.01 become different classes, while 7.01 and 9.8 become the same class.
- **Ranking.** Predict an order, and care only about the top of it. This is what a virtual screening campaign actually needs: the top 1% of a million-compound library is what gets bought. Metrics: enrichment factor, BEDROC, Spearman \(\rho\) on the top slice — Unit 19.

The threshold is a modelling decision with consequences, so look at what it does to the class balance before choosing it.



In [ ]:
## What a threshold does: class balance and the fraction of 'actives' as the cut-off moves.
ths <- seq(5, 8.5, by = 0.25)
bal <- data.table(threshold = ths,
                  n_active = sapply(ths, function(t) sum(y >= t)))
bal[, n_inactive := length(y) - n_active][, active_fraction := round(n_active / length(y), 3)]
print(ggplot(bal, aes(threshold, active_fraction)) +
  geom_hline(yintercept = 0.5, linetype = "dotted", colour = "grey40") +
  geom_line(colour = PAL["blue"]) + geom_point(colour = PAL["blue"], size = 1.4) +
  geom_text(data = bal[threshold %in% c(6, 7, 8)], aes(label = sprintf("%.2f", active_fraction)),
            nudge_y = 0.05, size = 2.9) +
  labs(x = "activity threshold (pActivity)", y = "fraction labelled 'active'",
       title = "Class balance is a consequence of the threshold"))
knitr::kable(bal[threshold %in% c(6, 6.5, 7, 7.5, 8)], caption = "Class balance at five candidate thresholds.")



<p class="caption"><b>Figure 2 and table.</b> Moving the activity threshold from 6.0 to 8.0 turns a 69%-active dataset into a 10%-active one: at pActivity ≥ 6.0 there are 1,898 actives against 866 inactives, at ≥ 7.0 the split is 914 against 1,850, and at ≥ 8.0 only 286 compounds remain active. Every classification metric in Unit 12 depends on this fraction — ROC-AUC least, precision and PR-AUC most — so the threshold must be reported with the metric, and classifiers trained at different thresholds are not comparable. For the classification examples of this block we use pActivity ≥ 7.0, that is 100 nM, which is both a round number and a plausible project-relevant potency gate.</p>

<div class="exercise"><h4>Exercise 1</h4>
<p>Re-run the chunk above with <code>ths &lt;- seq(4, 10, by = 0.1)</code> and find the threshold that balances the classes exactly. Then answer in one sentence why "the threshold that balances the classes" is almost never the right threshold scientifically, and what information you would need from a project chemist to choose it properly.</p></div>

## 2. The two feature families

A molecule is a graph, and no algorithm consumes graphs directly; Unit 5 of the cheminformatics course turns structures into vectors. Two families of vector dominate practice, and this block uses both because they have completely different statistics.

- **RDKit descriptors** — here eleven numbers per molecule (molecular weight, calculated lipophilicity MolLogP, topological polar surface area TPSA, counts of hydrogen-bond donors and acceptors, rotatable bonds, aromatic and total rings, heavy atoms, the fraction of sp³ carbons, and the composite drug-likeness score QED<sup class="c"><a href="#r25">[25]</a></sup>). They are dense, continuous, strongly correlated with one another and with molecular size, and each one is interpretable<sup class="c"><a href="#r27">[27]</a></sup>. They are already columns of `rock1_chembl_curated.csv`; we never recompute them.
- **Morgan fingerprints** (also called ECFP) — a 2,048-bit vector per molecule in which each bit records the presence of a circular atom environment of radius ≤ 2<sup class="c"><a href="#r10">[10]</a></sup>. They are sparse (about 2.5% of bits set per molecule), binary, high-dimensional, and individually meaningless but collectively a faithful record of substructure.

<div class="box warn"><h4>Substitution declared — where the fingerprints come from</h4>
<p>The Python edition computes Morgan fingerprints in the notebook with RDKit. There is no RDKit binding in this R environment (<code>rcdk</code> is not installable here), so the R edition <strong>loads the same fingerprints from a frozen sparse file</strong>, <code>data/rock1_morgan_sparse.csv.gz</code>: two columns, <code>id</code> (InChIKey) and <code>bit</code> (0-based index of a set bit), one row per set bit. They were generated once with RDKit 2026.03.4 and <code>rdFingerprintGenerator.GetMorganGenerator(radius = 2, fpSize = 2048)</code>, exactly as the Python unit does, and the recipe plus the SHA-256 of the file are recorded in <a href="../../data/manifest_fingerprints.json">data/manifest_fingerprints.json</a>. The structures themselves ship as <code>data/rock1_curated.sdf.gz</code>, so a reader with ChemmineR or rcdk can recompute the bits and check them. Reconstructing the matrix is one call to <code>Matrix::sparseMatrix()</code>, and Tanimoto similarity between its rows is three lines of sparse algebra, written in Section 4.</p></div>



In [ ]:
## Build the two feature matrices once; everything downstream reuses them.
DESC <- c("MolWt", "MolLogP", "TPSA", "HBD", "HBA", "RotB",
          "AromaticRings", "RingCount", "HeavyAtoms", "FractionCSP3", "QED")
X_desc <- as.matrix(reg[, ..DESC])

fpl  <- load_data("rock1_morgan_sparse.csv.gz")     ## (id, bit) pairs of the set bits
ikey <- reg$inchikey
fpl  <- fpl[id %in% ikey]
X_fp <- sparseMatrix(i = match(fpl$id, ikey), j = fpl$bit + 1L, x = 1,
                     dims = c(length(ikey), 2048),
                     dimnames = list(NULL, paste0("bit", 0:2047)))
stopifnot(nrow(X_fp) == nrow(X_desc), all(rowSums(X_fp) > 0))
cat("descriptors:", dim(X_desc), "| fingerprints:", dim(X_fp), "\n")
cat(sprintf("bits set per molecule: mean %.1f (%.2f%% of 2048) | bits never set in the whole set: %d\n",
            mean(rowSums(X_fp)), 100 * mean(X_fp), sum(colSums(X_fp) == 0)))
cat("columns (bits) present in at least 1% of compounds:", sum(colMeans(X_fp) >= 0.01), "\n")



The two matrices have the same number of rows and wildly different shapes: 11 dense columns against 2,048 sparse ones. Only 45 bits are never set here, but the distribution is extremely uneven: 719 bits occur in at least 1% of compounds and the rest are too rare to be useful. Unit 13 returns to this asymmetry in detail. The figure below shows both faces at once.



In [ ]:
## Left: how often each fingerprint bit is set. Right: the descriptor correlation matrix.
freq <- sort(colMeans(X_fp), decreasing = TRUE)
p1 <- ggplot(data.frame(rank = seq_along(freq), f = pmax(freq, 1e-4)), aes(rank, f)) +
  geom_line(colour = PAL["blue"]) +
  geom_hline(yintercept = 0.01, linetype = "dotted", colour = "grey40") +
  scale_y_log10() +
  labs(x = "bit, sorted by frequency", y = "fraction of compounds with the bit set",
       title = "Fingerprint bits are sparse and unequal")
C  <- cor(X_desc)
cm <- as.data.table(as.table(C)); setnames(cm, c("v1", "v2", "r"))
cm[, `:=`(v1 = factor(v1, levels = DESC), v2 = factor(v2, levels = rev(DESC)))]
p2 <- ggplot(cm, aes(v1, v2, fill = r)) + geom_tile() +
  scale_fill_gradient2(low = PAL["blue"], mid = "white", high = PAL["orange"], limits = c(-1, 1)) +
  labs(x = NULL, y = NULL, fill = "Pearson r", title = "Descriptors are dense and correlated") +
  theme(axis.text.x = element_text(angle = 90, hjust = 1, size = 6.5), axis.text.y = element_text(size = 6.5))
print(p1 | p2)
Coff <- C; diag(Coff) <- 0
ij <- which(abs(Coff) == max(abs(Coff)), arr.ind = TRUE)[1, ]
cat(sprintf("largest absolute descriptor correlation: %.3f between %s and %s\n",
            max(abs(Coff)), DESC[ij[1]], DESC[ij[2]]))



<p class="caption"><b>Figure 3.</b> The two feature families seen through their statistics. Left: fingerprint bit frequencies on a logarithmic scale, sorted — a handful of bits occur in most molecules, 719 in more than 1% of them (dotted line), and the rest are rare or absent; this long tail is what makes sparse binary data different from ordinary tabular data. Right: the correlation matrix of the eleven descriptors, whose orange block shows that molecular weight, heavy-atom count and hydrogen-bond acceptors measure nearly the same thing — the largest absolute correlation is 0.979, between HeavyAtoms and MolWt. Eleven descriptors are not eleven independent pieces of information, which matters for linear models (Unit 9) and for feature selection (Unit 13).</p>

## 3. Decisions 4 and 5 — the split and the preprocessing chain

### 3.1 The split, in one line and with one caveat

For the rest of this unit we hold out 20% of the compounds at random, fit everything on the remaining 80%, and touch the held-out fifth exactly twice: once in the error analysis of Section 7 and once in the model card of Section 9. Within the training 80% we use five-fold cross-validation for every comparison.

A random split is the *wrong* split for molecules, and Unit 12 shows by how much. We use it here because this unit is about preprocessing mechanics and the random split is the baseline against which the scaffold split is read; everything below is split-agnostic, and swapping the fold generator for a grouped one changes one line.



In [ ]:
## One held-out test set (20%) and one inner cross-validation scheme, used by every comparison below.
set.seed(0)
idx_te <- sort(sample(seq_along(y), round(0.20 * length(y))))
idx_tr <- setdiff(seq_along(y), idx_te)
set.seed(0)
folds  <- caret::createFolds(y[idx_tr], k = 5)       ## list of held-out row indices, 5 folds
cat("train:", length(idx_tr), "compounds | test:", length(idx_te), "compounds\n")
cat("fold sizes:", sapply(folds, length), "\n")
seen_scaf <- unique(reg$scaffold_smiles[idx_tr])
new_scaf  <- !(reg$scaffold_smiles[idx_te] %in% seen_scaf)
cat("test compounds whose Bemis-Murcko scaffold never appears in training:",
    sum(new_scaf), sprintf("(%.0f%%)\n", 100 * mean(new_scaf)))
cat("distinct scaffolds in the modelling subset:", uniqueN(reg$scaffold_smiles),
    "| singletons:", sum(table(reg$scaffold_smiles) == 1), "\n")



The output records something we will need twice. The modelling subset contains 1,218 distinct scaffolds — the ring framework that defines a chemical series<sup class="c"><a href="#r12">[12]</a></sup> — of which 858 occur exactly once. A random split therefore leaves 194 of the 553 test compounds (35%) with a scaffold the model has never seen, and the other 65% with a scaffold it has seen, often together with close analogues. Those two populations are not equally difficult, and reporting their average as one number hides the difference. Section 7 measures the gap.

### 3.2 Why a preprocessing chain, and what exactly goes wrong without one

Every preprocessing step that *learns something from the data* — a mean and a standard deviation for scaling, a median for imputation, a set of categories for dummy coding, a list of selected features, a PCA rotation — is a model with parameters, and those parameters must be estimated from the training fold alone. If you standardise the whole dataset before splitting, the test rows have contributed their mean to the transformation, and the evaluation is no longer blind. The effect is usually small for a simple scaler and large for anything supervised (feature selection, target encoding); Section 8 measures both.

R offers three idioms for the same guarantee, and this unit uses all three.

1. `caret::train(..., preProcess = c("center", "scale"))` — the preprocessing is attached to the model object and re-estimated inside every resample that `trainControl` generates<sup class="c"><a href="#r1">[1]</a></sup>. This is the closest analogue of a scikit-learn `Pipeline`.
2. A `recipes` recipe — a declarative list of `step_*` operations that is `prep`ared on training data and `bake`d onto new data. (`recipes` ships with the `caret` installation in this environment.)
3. A hand-written closure returning `fit` and `transform` functions, which is what you end up writing whenever the transformation is specific to your data (Section 3.4).



In [ ]:
## The minimal correct object: centre and scale, then a ridge regression - as ONE estimator.
tr_dat <- as.data.frame(X_desc[idx_tr, ]); tr_dat$y <- y[idx_tr]
set.seed(0)
fit_ridge <- caret::train(
  y ~ ., data = tr_dat, method = "glmnet", preProcess = c("center", "scale"),
  tuneGrid = expand.grid(alpha = 0, lambda = 10^seq(-2, 1, length.out = 12)),
  trControl = trainControl(method = "cv", number = 5))
best <- fit_ridge$results[which.min(fit_ridge$results$RMSE), ]
cat("the scaler learned its mean from", length(idx_tr), "training compounds only;\n")
cat(sprintf("  mean MolWt used by the transform : %.2f\n", fit_ridge$preProcess$mean[["MolWt"]]))
cat(sprintf("  mean MolWt of the full dataset   : %.2f  (never seen by this object)\n", mean(X_desc[, "MolWt"])))
cat(sprintf("selected ridge penalty lambda = %.4f | 5-fold CV RMSE = %.3f | R2 = %.3f\n",
            best$lambda, best$RMSE, best$Rsquared))
r2 <- function(obs, pred) 1 - sum((obs - pred)^2) / sum((obs - mean(obs))^2)
cat(sprintf("train R2 = %.3f | test R2 = %.3f\n",
            r2(y[idx_tr], predict(fit_ridge, as.data.frame(X_desc[idx_tr, ]))),
            r2(y[idx_te], predict(fit_ridge, as.data.frame(X_desc[idx_te, ])))))



The two printed means differ in the third significant digit (382.03 against 381.59): scaling is unsupervised and 2,211 compounds estimate a mean well. Use the chain anyway, because the *discipline* generalises — the object that protects you from a leaky scaler protects you from a leaky feature selector, where the optimism is not trivial at all. Note also the scores: a cross-validated \(R^2\) of 0.124 and a test \(R^2\) of 0.102, against a training \(R^2\) of 0.132. A model whose training and test scores are this close is not overfitting; it is simply not learning much, and Section 6 will say why.

### 3.3 Recipes: tables whose columns are not all the same kind

Real tables mix numeric columns, categorical columns and missing values. A `recipes` recipe is the R counterpart of scikit-learn's `ColumnTransformer`: each `step_*` declares an operation on a selection of columns, `prep()` estimates whatever has to be estimated (medians, means, category levels) from the training data, and `bake()` applies the frozen result to new data. The example below builds a mixed table from the ROCK1 data: the eleven numeric descriptors, the mostly-missing clinical-phase column (`max_phase`), and a categorical column recording which assay type dominates the compound's measurements — IC50, Ki or Kd, a genuine source of systematic offset between records<sup class="c"><a href="#r13">[13]</a></sup>.



In [ ]:
## A mixed-type table: numeric descriptors (imputed + scaled), a sparse numeric column, one categorical.
tab <- as.data.table(reg[, c(..DESC, "max_phase")])
tab[, assay_family := fifelse(reg$n_IC50 >= pmax(reg$n_Ki, reg$n_Kd), "IC50",
                      fifelse(reg$n_Ki >= reg$n_Kd, "Ki", "Kd"))]
tab[, y := y]
rec <- recipe(y ~ ., data = tab[idx_tr]) |>
  step_impute_median(all_numeric_predictors()) |>
  step_normalize(all_numeric_predictors()) |>
  step_dummy(all_nominal_predictors(), one_hot = TRUE)
prepped <- prep(rec, training = tab[idx_tr], verbose = FALSE)
cat("assay_family counts:", paste(names(table(tab$assay_family)), table(tab$assay_family),
                                 sep = "=", collapse = "  "), "\n")
cat("max_phase missing in", sum(is.na(tab$max_phase)), "of", nrow(tab),
    "rows -> imputed with the TRAINING median\n")
cat("columns after baking:", paste(tail(names(bake(prepped, new_data = tab[idx_te])), 4), collapse = ", "), "\n")
set.seed(0)
fit_mixed <- caret::train(rec, data = as.data.frame(tab[idx_tr]), method = "glmnet",
                          tuneGrid = expand.grid(alpha = 0, lambda = 10^seq(-2, 1, length.out = 8)),
                          trControl = trainControl(method = "cv", number = 5))
bm <- fit_mixed$results[which.min(fit_mixed$results$RMSE), ]
cat(sprintf("5-fold CV on the mixed table: RMSE = %.3f | R2 = %.3f\n", bm$RMSE, bm$Rsquared))



The mixed table reaches a cross-validated \(R^2\) of 0.132 and an RMSE of 1.026 log units — barely better than predicting the mean — which is the honest verdict on eleven bulk descriptors plus two administrative columns as a model of kinase potency. The point of the chunk is not the score but the mechanics: `caret::train()` accepts the recipe itself, so the median that fills the 2,717 missing `max_phase` values and the means and standard deviations of `step_normalize` are re-estimated inside each of the five resamples, the dummy coding of `assay_family` produces one column per level, and the printed column names let you trace every column of the final matrix back to its origin — the single most useful debugging habit with composite preprocessing.

### 3.4 A hand-written transformer: featurisation inside the fold

Descriptors and fingerprints are themselves preprocessing. For a *deterministic, row-wise* featurisation such as a Morgan fingerprint, computing it once before the split is safe — the fingerprint of a molecule does not depend on any other molecule. The habit becomes dangerous the moment the featurisation learns from the set as a whole: a bit filter ("keep bits present in at least 1% of compounds"), a scaler, a PCA, a learned embedding. Those must be fitted inside the fold.

In R the lightest way to express "an object with `fit` and `transform`" is a closure: a function that captures what it learned and returns a function that applies it. The transformer below keeps only the fingerprint bits that are frequent enough *in the rows it was fitted on*, and the manual cross-validation beneath it calls `fit` separately in every fold.



In [ ]:
## fit/transform as a closure: a rare-bit filter learned from the training fold only.
make_bit_filter <- function(min_freq = 0.01) {
  list(fit = function(X) {
         keep <- which(Matrix::colMeans(X) >= min_freq)
         list(keep = keep, transform = function(Xnew) Xnew[, keep, drop = FALSE])
       })
}
rmse <- function(obs, pred) sqrt(mean((obs - pred)^2))
mae  <- function(obs, pred) mean(abs(obs - pred))

bf  <- make_bit_filter(0.01)
kept_per_fold <- integer(0); r2_per_fold <- numeric(0)
for (te in folds) {
  tr  <- setdiff(seq_along(idx_tr), te)
  f   <- bf$fit(X_fp[idx_tr[tr], ])                       ## fitted on the training fold only
  Xtr <- f$transform(X_fp[idx_tr[tr], ]); Xte <- f$transform(X_fp[idx_tr[te], ])
  m   <- glmnet(as.matrix(Xtr), y[idx_tr[tr]], alpha = 0, lambda = 0.05)
  p   <- as.numeric(predict(m, as.matrix(Xte)))
  kept_per_fold <- c(kept_per_fold, length(f$keep))
  r2_per_fold   <- c(r2_per_fold, r2(y[idx_tr[te]], p))
}
cat("bits kept in each fold (>= 1% frequency):", kept_per_fold, "\n")
cat(sprintf("ridge on filtered fingerprints, 5-fold CV R2 = %.3f (folds: %s)\n",
            mean(r2_per_fold), paste(round(r2_per_fold, 3), collapse = ", ")))



The filter keeps between 719 and 736 of the 2,048 bits depending on the fold — the number itself is data-dependent, which is precisely why it must be re-estimated — and the cross-validated \(R^2\) is 0.421, about three and a half times the descriptor model's 0.124. A chain of this shape is what should be serialised with `saveRDS()` at the end of a project: the only object that reproduces the full transformation from raw data to prediction.

<div class="box key"><h4>Key idea</h4>
<p>The preprocessing chain is not a convenience; it is the formal statement of what your model <em>is</em>. If the artefact is "a ridge regression plus a scaler from chunk 12 plus a bit filter from chunk 7", nobody — including you in six months — can reproduce the prediction. If it is one <code>caret</code> model, one recipe or one closure, then fitting, resampling and tuning all do the right thing automatically. Kuhn and Johnson's <em>Applied Predictive Modeling</em><sup class="c"><a href="#r31">[31]</a></sup> is the book-length version of this section.</p></div>

## 4. Baselines, and the only honest way to report a score

An \(R^2\) of 0.33 means nothing by itself. Compared to what? Three baselines belong in every supervised project, and they cost seconds:

- **The constant baseline.** Predict the training mean for every compound, or the majority class for every patient. Its RMSE is the standard deviation of the target; its \(R^2\) is 0 by construction. Any model that does not beat it is not a model.
- **The nearest-neighbour baseline.** Predict the activity of the single most similar training compound. In chemistry this is a strong baseline and a meaningful one: it is the quantitative form of "look up the closest analogue in the series", which is what a medicinal chemist does without any model at all<sup class="c"><a href="#r22">[22]</a></sup>.
- **The linear baseline.** A ridge regression<sup class="c"><a href="#r30">[30]</a></sup> on the same features. If a random forest cannot beat a regularised linear model, the extra complexity is not earning its keep.

The table below computes all three on both feature families, plus a random forest<sup class="c"><a href="#r26">[26]</a></sup><sup class="c"><a href="#r20">[20]</a></sup> on each, inside the same five-fold cross-validation of the training set. The nearest-neighbour model on fingerprints uses the Tanimoto (Jaccard) coefficient, the standard similarity measure for binary substructure vectors; since no R package here implements a Tanimoto \(k\)-nearest-neighbour regressor, we write it in three lines of sparse algebra. For two binary vectors with \(a\) and \(b\) bits set and \(c\) bits in common,

\[ T(x_1, x_2) = \frac{c}{a + b - c}, \]

and \(c\) for all pairs at once is the matrix product \(X_1 X_2^{\top}\).



In [ ]:
## Tanimoto similarity between the rows of two sparse binary matrices, and two 1-NN regressors.
tanimoto <- function(A, B) {
  C <- as.matrix(Matrix::tcrossprod(A, B))               ## shared bits, all pairs at once
  a <- Matrix::rowSums(A); b <- Matrix::rowSums(B)
  C / (outer(a, b, "+") - C)
}
knn1_tanimoto <- function(Xtr, ytr, Xte) ytr[apply(tanimoto(Xte, Xtr), 1, which.max)]
knn1_euclid <- function(Xtr, ytr, Xte) {               ## scaled with TRAINING statistics only
  m <- colMeans(Xtr); s <- apply(Xtr, 2, sd)
  A <- scale(Xtr, m, s); B <- scale(Xte, m, s)
  d <- as.matrix(dist(rbind(B, A)))[seq_len(nrow(B)), nrow(B) + seq_len(nrow(A))]
  ytr[apply(d, 1, which.min)]
}
## A worked check of the formula on the first two compounds.
A <- X_fp[1, , drop = FALSE]; B <- X_fp[2, , drop = FALSE]
cat(sprintf("compound 1 has %d bits, compound 2 has %d, they share %d -> T = %d/(%d+%d-%d) = %.4f (function: %.4f)\n",
            sum(A), sum(B), sum(A * B), sum(A * B), sum(A), sum(B), sum(A * B),
            sum(A * B) / (sum(A) + sum(B) - sum(A * B)), tanimoto(A, B)[1, 1]))



The printed line confirms the three-line implementation against the definition on one pair of molecules. With the similarity in hand, the comparison table follows.

<div class="box warn"><h4>An R trap worth knowing</h4>
<p><code>ranger()</code> treats <code>seed = 0</code> as "no seed given, draw one at random": a script that passes <code>seed = 0</code> produces a <em>different</em> forest on every run. The drift is not cosmetic: in a test of this very model, individual held-out predictions moved by up to 0.12 log units between two runs of identical code, and aggregate scores such as RMSE or \(R^2\) wander in the second or third decimal — enough to change which model "wins" a close comparison. Every <code>ranger()</code> call in this document therefore passes <code>seed = 1</code> and an explicit <code>num.threads</code>, which makes the whole unit reproducible. The equivalent trap in <code>xgboost</code> (Unit 14) is <code>nthread</code>, and in <code>Rtsne</code> (Unit 13) it is forgetting <code>set.seed()</code> before the call.</p></div>




In [ ]:
## Baselines and two real models, all under the same 5-fold CV of the training set. (~20 s)
ytr <- y[idx_tr]; Xtr_d <- X_desc[idx_tr, ]; Xtr_f <- X_fp[idx_tr, ]
f_mean  <- function(Xtr, yy, Xte) rep(mean(yy), nrow(Xte))
f_ridge <- function(Xtr, yy, Xte) {                      ## penalty re-selected inside every fold
  set.seed(1); cv <- cv.glmnet(as.matrix(Xtr), yy, alpha = 0, nfolds = 5)
  as.numeric(predict(cv, as.matrix(Xte), s = "lambda.min"))
}
f_rf <- function(Xtr, yy, Xte) {
  m <- ranger(x = as.matrix(Xtr), y = yy, num.trees = 300, seed = 1, num.threads = 4)
  predict(m, as.matrix(Xte))$predictions
}
cv_eval <- function(fitpred, X, yy, folds) {
  per <- t(sapply(folds, function(te) {
    tr <- setdiff(seq_along(yy), te)
    p  <- fitpred(X[tr, , drop = FALSE], yy[tr], X[te, , drop = FALSE])
    c(RMSE = rmse(yy[te], p), MAE = mae(yy[te], p), R2 = r2(yy[te], p))
  }))
  c(colMeans(per), RMSE_sd = sd(per[, "RMSE"]))
}
cands <- list(list("mean (constant baseline)",       f_mean,        Xtr_d),
              list("1-NN, descriptors (Euclidean)",  knn1_euclid,   Xtr_d),
              list("ridge, descriptors",             f_ridge,       Xtr_d),
              list("random forest, descriptors",     f_rf,          Xtr_d),
              list("1-NN, fingerprints (Tanimoto)",  knn1_tanimoto, Xtr_f),
              list("ridge, fingerprints",            f_ridge,       Xtr_f),
              list("random forest, fingerprints",    f_rf,          Xtr_f))
base <- rbindlist(lapply(cands, function(cc) {
  t0 <- Sys.time(); m <- cv_eval(cc[[2]], cc[[3]], ytr, folds)
  data.table(model = cc[[1]], RMSE = m[["RMSE"]], RMSE_sd = m[["RMSE_sd"]],
             MAE = m[["MAE"]], R2 = m[["R2"]],
             fit_s = as.numeric(difftime(Sys.time(), t0, units = "secs")) / 5)
}))
base[, RMSE_vs_mean_pct := round(100 * (1 - RMSE / RMSE[1]), 1)]
knitr::kable(base[, lapply(.SD, function(v) if (is.numeric(v)) round(v, 3) else v)],
             caption = "Five-fold cross-validated performance on the 2,211 training compounds.")



<p class="caption"><b>Table (Section 4).</b> Five-fold cross-validated performance on the training compounds. The constant baseline has RMSE 1.099 (the target's standard deviation) and \(R^2\) ≈ 0. Reading down the table: a 1-nearest-neighbour model on descriptors is <em>worse</em> than the mean (RMSE 1.128, \(R^2\) = −0.059) — bulk properties do not identify a useful analogue; ridge on descriptors improves little (\(R^2\) = 0.124); the random forest on descriptors reaches \(R^2\) = 0.324; and on fingerprints, 1-NN by Tanimoto similarity alone achieves \(R^2\) = 0.243, ridge 0.440 and the random forest 0.543, with RMSE 0.742, a 32.4% reduction over the constant baseline. The last column is the number to report in a paper: "the model reduces RMSE by 32% relative to predicting the mean", not a bare \(R^2\). Note also the cost column — the cross-validated ridge on 2,048 bits spends about two seconds per fold re-selecting its penalty and the forest about one, against milliseconds for the baselines, which is the exchange rate between accuracy and compute you will negotiate again in Unit 14.</p>



In [ ]:
## The same table as a figure, with the baselines marked.
bp <- copy(base)[, kind := ifelse(grepl("forest", model), "model", "baseline")]
bp[, model := factor(model, levels = model[order(RMSE, decreasing = TRUE)])]
print(ggplot(bp, aes(model, RMSE, fill = kind)) +
  geom_col(width = 0.62) +
  geom_errorbar(aes(ymin = RMSE - RMSE_sd, ymax = RMSE + RMSE_sd), width = 0.2, linewidth = 0.3) +
  geom_hline(yintercept = base$RMSE[1], linetype = "dashed", linewidth = 0.4) +
  scale_fill_manual(values = c(baseline = "grey60", model = unname(PAL["blue"])), guide = "none") +
  coord_flip() +
  labs(x = NULL, y = "cross-validated RMSE (log units of pActivity; lower is better)",
       title = "Every model, against the baselines"))
cat("per-fold RMSE of the random forest on fingerprints:",
    round(sapply(folds, function(te) {
      tr <- setdiff(seq_along(ytr), te); rmse(ytr[te], f_rf(Xtr_f[tr, ], ytr[tr], Xtr_f[te, ]))
    }), 3), "\n")



<p class="caption"><b>Figure 4.</b> The baseline comparison as a figure; grey bars are baselines, blue bars the two random forests, error bars the standard deviation across the five folds, dashed line the constant baseline at 1.099. Two lessons are visible at a glance. First, the largest single jump comes from changing the <em>representation</em> (descriptors → fingerprints) rather than the algorithm: the same forest moves from 0.902 to 0.742 RMSE, and even a plain ridge on fingerprints (0.822) beats the forest on descriptors. The choice of features dominates the choice of model, which is why Unit 13 exists. Second, the fold-to-fold standard deviation — the printed per-fold values of the best model are 0.773, 0.750, 0.730, 0.703 and 0.756, a standard deviation of 0.027 — is the yardstick for judging whether two bars differ: differences smaller than that are noise, and Unit 12 turns that intuition into a test.</p>

### 4.1 The sanity check that costs one line

Before believing any chain, shuffle the labels and re-run it. With the correspondence between molecules and activities destroyed, an honest evaluation must return \(R^2\) ≈ 0 (in practice slightly negative) and RMSE ≈ the target standard deviation. If it does not, something in the chain is reading the target.



In [ ]:
## Sanity check: destroy the structure-activity relationship and verify the chain reports nothing.
set.seed(0); y_shuffled <- sample(ytr)
real <- cv_eval(f_ridge, Xtr_d, ytr,        folds)
shuf <- cv_eval(f_ridge, Xtr_d, y_shuffled, folds)
cat(sprintf("ridge on descriptors, real labels     : R2 = %+.3f | RMSE = %.3f\n", real[["R2"]], real[["RMSE"]]))
cat(sprintf("ridge on descriptors, shuffled labels : R2 = %+.3f | RMSE = %.3f\n", shuf[["R2"]], shuf[["RMSE"]]))
cat("fold-by-fold with shuffled labels     :",
    round(sapply(folds, function(te) {
      tr <- setdiff(seq_along(ytr), te)
      r2(y_shuffled[te], f_ridge(Xtr_d[tr, ], y_shuffled[tr], Xtr_d[te, ]))
    }), 3), "\n")



With shuffled labels the cross-validated \(R^2\) is −0.002 and every individual fold lies within 0.007 of zero, while the RMSE returns to the target's standard deviation: the chain is honest. A *negative* \(R^2\) is the correct outcome, not a bug — it means the model predicts worse than the mean on held-out data, which is what a model fitted to noise must do. Unit 12 turns this one-line check into the formal y-randomisation test with a null distribution of 100 scrambles<sup class="c"><a href="#r8">[8]</a></sup>.

## 5. A worked example by hand: what the regression metrics are

Before trusting a metric, compute it once on paper. Take five compounds from the held-out set with observed pActivity \(y_i\) and predictions \(\hat{y}_i\) rounded to two decimals, and let \(\bar{y}\) be the mean of the five observed values:

\[ \mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_i (y_i-\hat{y}_i)^2}, \qquad
   \mathrm{MAE} = \frac{1}{n}\sum_i |y_i-\hat{y}_i|, \qquad
   R^2 = 1 - \frac{\sum_i (y_i-\hat{y}_i)^2}{\sum_i (y_i-\bar{y})^2}. \]

With the five rows printed by the next chunk, the arithmetic is: sum the squared residuals to get the *residual sum of squares* (RSS), sum the squared deviations from the mean to get the *total sum of squares* (TSS), then RMSE = \(\sqrt{\mathrm{RSS}/5}\), MAE = \(\sum|\text{residual}|/5\) and \(R^2 = 1 - \mathrm{RSS}/\mathrm{TSS}\). By hand, with \(\bar y = 5.817\): the residuals are \(+0.05, +0.70, -0.55, -0.86, -0.41\), so RSS = \(0.0025+0.4900+0.3025+0.7396+0.1681 = 1.7027\) and TSS = \(0.3394+0.7667+0.5190+0.5190+0.0003 = 2.1444\); hence RMSE = \(\sqrt{1.7027/5}\) = 0.584, MAE = \(2.57/5\) = 0.514 and \(R^2 = 1-1.7027/2.1444 = 0.206\). The chunk recomputes the same quantities from the unrounded residuals, which is why its RSS is 1.7154, its RMSE 0.5857 and its \(R^2\) 0.2000 rather than exactly the hand figures — a useful reminder that rounding before aggregating moves the third decimal.



In [ ]:
## Worked example: five held-out compounds, the three regression metrics by hand and by function.
set.seed(0)
rf_fin  <- ranger(x = as.matrix(Xtr_f), y = ytr, num.trees = 300, seed = 1, num.threads = 4)
five    <- idx_te[1:5]
yv      <- y[five]
pv      <- round(predict(rf_fin, as.matrix(X_fp[five, ]))$predictions, 2)
hand <- data.frame(y = round(yv, 2), y_hat = pv, residual = round(yv - pv, 2),
                   `residual^2` = round((yv - pv)^2, 4),
                   `(y-ybar)^2` = round((yv - mean(yv))^2, 4), check.names = FALSE)
print(hand)
RSS <- sum((yv - pv)^2); TSS <- sum((yv - mean(yv))^2)
cat(sprintf("\nybar = %.3f | RSS = %.4f | TSS = %.4f\n", mean(yv), RSS, TSS))
cat(sprintf("by hand  : RMSE = %.4f | MAE = %.4f | R2 = %.4f\n",
            sqrt(RSS / 5), mean(abs(yv - pv)), 1 - RSS / TSS))
cat(sprintf("by helper: RMSE = %.4f | MAE = %.4f | R2 = %.4f\n",
            rmse(yv, pv), mae(yv, pv), r2(yv, pv)))



The two computed lines agree to four decimals, and the hand arithmetic above agrees to the second: nothing in a metric function is mysterious. Three properties follow from the arithmetic. RMSE squares the residuals, so one compound wrong by 2 log units contributes as much as sixteen wrong by 0.5 — it is a statement about the worst cases. MAE weights every compound equally and is the honest "typical error"; here 0.52 against an RMSE of 0.59. \(R^2\) is *relative*: its denominator is the variance of the set you computed it on, so the same model scores differently on a diverse and on a narrow test set, and \(R^2\) on five compounds (0.200 here, against 0.585 on the full test set) is meaningless. Report RMSE and MAE in log units, which the chemist compares with assay reproducibility, and treat \(R^2\) as a secondary, set-dependent summary<sup class="c"><a href="#r8">[8]</a></sup>.

## 6. Learning curves: more data or more model?

A learning curve plots training and validation error against the number of training examples. It answers the question that every project asks at month three: *should I spend the next month measuring more compounds, or building a better model?*

- If the validation curve is still falling at the right-hand edge, more data will help.
- If the two curves have converged and both sit high, the model is **underfitting** (high bias): more data will not help, more capacity or better features might.
- If the training error is near zero and the gap to the validation error is wide and not closing, the model is **overfitting** (high variance)<sup class="c"><a href="#r23">[23]</a></sup>: more data, stronger regularisation or fewer features will help.

R has no single `learning_curve()` function, so we write the loop; fifteen lines, and you see exactly what is resampled. The two chunks below draw the curve for the two extremes of our table: the random forest on fingerprints (flexible, 2,048 features) and ridge on descriptors (rigid, 11 features).



In [ ]:
## Learning curve: 3-fold CV at five training sizes, training error measured on the subsample used. (~15 s)
learning_curve <- function(X, yy, fitpred, fracs = c(0.15, 0.36, 0.57, 0.79, 1.0), k = 3) {
  set.seed(0); fo <- caret::createFolds(yy, k = k)
  nmin <- min(sapply(fo, function(te) length(yy) - length(te)))
  out <- list()
  for (f in fracs) {
    nsub <- floor(f * nmin)
    for (i in seq_along(fo)) {
      te <- fo[[i]]; tr <- setdiff(seq_along(yy), te)
      set.seed(100 + i); sub <- sample(tr, nsub)
      out[[length(out) + 1]] <- data.table(
        n = nsub, fold = i,
        train = rmse(yy[sub], fitpred(X[sub, , drop = FALSE], yy[sub], X[sub, , drop = FALSE])),
        valid = rmse(yy[te],  fitpred(X[sub, , drop = FALSE], yy[sub], X[te,  , drop = FALSE])))
    }
  }
  rbindlist(out)[, .(train = mean(train), valid = mean(valid), sd_valid = sd(valid)), by = n]
}
f_rf150 <- function(Xtr, yy, Xte) {
  m <- ranger(x = as.matrix(Xtr), y = yy, num.trees = 150, seed = 1, num.threads = 4)
  predict(m, as.matrix(Xte))$predictions
}
lc_rf <- learning_curve(X_fp, y, f_rf150)
lcp <- melt(lc_rf, id.vars = "n", measure.vars = c("train", "valid"))
print(ggplot(lcp, aes(n, value, colour = variable)) +
  geom_line() + geom_point(size = 1.4) +
  geom_hline(yintercept = sd(y), linetype = "dashed", linewidth = 0.4) +
  scale_colour_manual(values = c(train = unname(PAL["orange"]), valid = unname(PAL["blue"])),
                      labels = c("training RMSE", "validation RMSE"), name = NULL) +
  labs(x = "training compounds", y = "RMSE (log units)",
       title = "Random forest on 2,048-bit fingerprints") + theme(legend.position = "top"))
print(lc_rf[, lapply(.SD, round, 3)])



<p class="caption"><b>Figure 5.</b> Learning curve of the random forest on fingerprints. The training RMSE stays between 0.428 and 0.463 at every size — the forest always fits its own training set well — while the validation RMSE falls from 0.907 at 276 compounds to 0.741 at 1,842 and is <em>still falling</em> at the right-hand edge, by about 0.02 log units over the last step. The large persistent gap (0.313 at the largest size) is the signature of a high-variance model, and the downward slope is the practical message: for this target, measuring more compounds would still buy accuracy. The dashed line is the constant baseline (1.109); even the smallest training set beats it.</p>



In [ ]:
## The opposite regime: a rigid model on eleven descriptors.
f_ridge_fixed <- function(Xtr, yy, Xte) {
  m <- glmnet(as.matrix(Xtr), yy, alpha = 0, lambda = 0.05, standardize = TRUE)
  as.numeric(predict(m, as.matrix(Xte)))
}
lc_ri <- learning_curve(X_desc, y, f_ridge_fixed)
lcp2 <- melt(lc_ri, id.vars = "n", measure.vars = c("train", "valid"))
print(ggplot(lcp2, aes(n, value, colour = variable)) +
  geom_line() + geom_point(size = 1.4) +
  geom_hline(yintercept = sd(y), linetype = "dashed", linewidth = 0.4) +
  scale_colour_manual(values = c(train = unname(PAL["orange"]), valid = unname(PAL["blue"])),
                      labels = c("training RMSE", "validation RMSE"), name = NULL) +
  ylim(0.3, 1.2) +
  labs(x = "training compounds", y = "RMSE (log units)",
       title = "Ridge on 11 descriptors") + theme(legend.position = "top"))
cat("gap at the largest size: ridge", round(lc_ri$valid[nrow(lc_ri)] - lc_ri$train[nrow(lc_ri)], 3),
    "| forest", round(lc_rf$valid[nrow(lc_rf)] - lc_rf$train[nrow(lc_rf)], 3), "\n")
print(lc_ri[, lapply(.SD, round, 3)])



<p class="caption"><b>Figure 6.</b> The same experiment for ridge on eleven descriptors. Here the two curves are almost on top of each other at about 1.04 RMSE from the first training size onwards: the gap at the largest size is 0.006 log units, against 0.313 for the forest, and the validation curve improves by 0.008 in total between 276 and 1,842 compounds. This is textbook underfitting — the model has already extracted everything eleven bulk properties contain, and no amount of extra data will move it. The decision implied by the two figures is therefore asymmetric: for the descriptor model, invest in features; for the fingerprint forest, invest in data.</p>

<div class="exercise"><h4>Exercise 2</h4>
<p>Redraw Figure 5 with <code>ranger(..., max.depth = 4)</code> and with <code>ranger(..., min.node.size = 20)</code>. Which of the three settings has the smallest gap between the curves, which has the lowest validation RMSE, and are they the same setting? Write one sentence explaining, in terms of bias and variance, why they need not be.</p></div>

## 7. Error analysis: which compounds are wrong, and why

A single test metric is the beginning of the analysis, not the end. The questions that decide whether a model is usable are: *where* does it fail, is the failure systematic, and does it fail on the compounds you care about? We now use the held-out fifth for the first time.



In [ ]:
## The held-out test set, scored once, with four metrics and the full prediction vector kept.
pred_te <- predict(rf_fin, as.matrix(X_fp[idx_te, ]))$predictions
cat(sprintf("held-out test set (n = %d compounds, never used until now)\n", length(idx_te)))
cat(sprintf("  RMSE      = %.3f log units   (constant baseline: %.3f)\n", rmse(y[idx_te], pred_te), sd(ytr)))
cat(sprintf("  MAE       = %.3f log units\n", mae(y[idx_te], pred_te)))
cat(sprintf("  R2        = %.3f\n", r2(y[idx_te], pred_te)))
cat(sprintf("  Spearman  = %.3f   (how well the ranking is reproduced)\n",
            cor(y[idx_te], pred_te, method = "spearman")))



The held-out numbers — RMSE 0.738 log units, \(R^2\) 0.585, Spearman \(\rho\) 0.774 — are close to the cross-validated estimate of Section 4 (RMSE 0.742): a single test set is itself a random draw, and with 553 compounds a difference of 0.004 log units is well inside the fold-to-fold noise of 0.027. Unit 12 puts a confidence interval on that number instead of quoting it bare. Spearman \(\rho\) 0.774 against \(R^2\) 0.585 says something else: the model orders compounds better than it places them, and for a project that uses it to *rank* candidates the ranking correlation is the more relevant figure.

### 7.1 The ranked error table

The standard diagnostic is a table of the worst predictions annotated with everything you know about those compounds: how many measurements stand behind the target, the replicate spread, molecular size and lipophilicity, whether the scaffold appeared in training, and how similar the nearest training compound is.



In [ ]:
## Nearest-neighbour similarity of every test compound to the training set, then the ranked error table.
max_sim <- apply(tanimoto(X_fp[idx_te, ], Xtr_f), 1, max)
err <- data.table(name = reg$pref_name[idx_te], y = y[idx_te], pred = pred_te,
                  residual = y[idx_te] - pred_te, n_meas = reg$n_measurements[idx_te],
                  spread = reg$pActivity_range[idx_te], MolWt = reg$MolWt[idx_te],
                  MolLogP = reg$MolLogP[idx_te], max_sim_train = max_sim, new_scaffold = new_scaf)
err[, abs_err := abs(residual)]
cat(sprintf("median nearest-neighbour Tanimoto similarity to training: %.3f\n", median(max_sim)))
worst <- head(err[order(-abs_err)], 12)
knitr::kable(worst[, .(y = round(y, 2), pred = round(pred, 2), residual = round(residual, 2),
                       n_meas, spread, MolWt = round(MolWt), MolLogP = round(MolLogP, 1),
                       max_sim_train = round(max_sim_train, 2), new_scaffold)],
             caption = "The twelve worst-predicted held-out compounds.")
cat("of these twelve,", sum(worst$n_meas == 1), "rest on a single measurement and",
    sum(worst$new_scaffold), "carry a scaffold absent from training\n")



<p class="caption"><b>Table (Section 7.1).</b> The twelve worst-predicted held-out compounds. Three patterns are visible and all three are chemically meaningful. (i) Every one of them is wrong by between 1.84 and 2.68 log units — a factor of 70 to 500 in potency — while the typical error is 0.57; the error distribution has a heavy tail rather than being uniformly mediocre. (ii) Ten of the twelve rest on a <em>single</em> measurement (<code>n_meas</code> = 1, <code>spread</code> = 0), so part of what we are calling model error is unreplicated experimental error. (iii) Six carry a scaffold absent from training, and the sign of the error splits cleanly: compounds observed at pActivity 9 or above are predicted around 7.6–7.9, and compounds observed near 4–5 are predicted around 6–7. The largest error of all (observed 4.00, predicted 6.68) is the most instructive: its nearest training neighbour has Tanimoto similarity 1.00, i.e. an identical Morgan fingerprint with a very different measured potency. That is an <strong>activity cliff</strong><sup class="c"><a href="#r28">[28]</a></sup> in its purest form — the irreducible failure mode of similarity-based prediction, and a reminder that two molecules with the same circular substructures are not the same molecule.</p>



In [ ]:
## Where the errors live: observed against predicted, and the error distribution.
p1 <- ggplot(err, aes(pred, y)) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", linewidth = 0.4) +
  geom_point(size = 0.8, alpha = 0.45, colour = PAL["blue"]) +
  coord_equal(xlim = range(err$y) + c(-0.3, 0.3), ylim = range(err$y) + c(-0.3, 0.3)) +
  labs(x = "predicted pActivity", y = "observed pActivity",
       title = sprintf("Observed vs predicted (RMSE %.2f)", rmse(y[idx_te], pred_te)))
p2 <- ggplot(err, aes(residual)) +
  geom_histogram(bins = 40, fill = PAL["green"], alpha = 0.85) +
  geom_vline(xintercept = 0, linewidth = 0.4) +
  labs(x = "residual (observed - predicted)", y = "compounds", title = "Residual distribution")
print(p1 | p2)
cat(sprintf("mean residual for weak compounds   (y < 5.5, n = %d): %+.2f\n",
            sum(err$y < 5.5), mean(err[y < 5.5, residual])))
cat(sprintf("mean residual for potent compounds (y > 7.5, n = %d): %+.2f\n",
            sum(err$y > 7.5), mean(err[y > 7.5, residual])))



<p class="caption"><b>Figure 7.</b> Left: observed against predicted pActivity for the 553 held-out compounds, with the identity line. The cloud is tilted — weak compounds are predicted too potent (mean residual −0.86 below pActivity 5.5, over 110 compounds) and potent ones too weak (+0.73 above 7.5, over 111 compounds). This is <strong>regression to the mean</strong>: a squared-error model shrinks predictions towards the centre, most strongly where data are thinnest, so the model is a ranking device rather than a potency calculator at the extremes. Right: the residual distribution, centred on zero, with the heavy tails documented above; note that its visible width is about ±1.5 log units while its standard deviation is 0.74.</p>

### 7.2 Error as a function of distance to the training set

The single most useful slice of the error is by similarity to the training set, because it tells you whether the model should be trusted on a *new* compound at all.



In [ ]:
## RMSE by quartile of nearest-neighbour similarity, and by whether the scaffold was seen in training.
err[, simq := cut(max_sim_train, quantile(max_sim_train, 0:4 / 4), include.lowest = TRUE,
                  labels = c("Q1 (most dissimilar)", "Q2", "Q3", "Q4 (most similar)"))]
byq <- err[, .(n = .N, RMSE = sqrt(mean(residual^2)), median_sim = median(max_sim_train)), by = simq][order(simq)]
print(ggplot(byq, aes(simq, RMSE, fill = simq)) + geom_col(width = 0.62) +
  geom_text(aes(label = sprintf("n = %d\n%.2f", n, RMSE)), vjust = -0.2, size = 2.8) +
  scale_fill_manual(values = unname(PAL[c("orange", "orange", "blue", "blue")]), guide = "none") +
  ylim(0, 0.95) +
  labs(x = NULL, y = "RMSE (log units)",
       title = "Error grows as the nearest training neighbour gets further away"))
print(byq[, lapply(.SD, function(v) if (is.numeric(v)) round(v, 3) else v)])
cat(sprintf("RMSE, scaffold seen in training (n = %d) : %.3f\n",
            sum(!new_scaf), sqrt(mean(err[new_scaffold == FALSE, residual^2]))))
cat(sprintf("RMSE, scaffold NEW to the model (n = %d) : %.3f\n",
            sum(new_scaf), sqrt(mean(err[new_scaffold == TRUE, residual^2]))))



<p class="caption"><b>Figure 8.</b> Held-out RMSE by quartile of nearest-neighbour Tanimoto similarity to the training set. The two most dissimilar quartiles (median similarity 0.56 and 0.73) are predicted with RMSE 0.802 and 0.780, the two most similar (median 0.80 and 0.89) with 0.681 and 0.680 — a 15% difference purely as a function of how close the nearest analogue is. The printed lines make a weaker version of the same point with scaffolds: RMSE 0.725 for ring frameworks already in training against 0.761 for new ones. <strong>This comparison is the argument of Unit 12 in miniature</strong>: a random split mixes the two populations and reports their average, while the real question — "will this model work on a chemotype we have not seen?" — is answered only by the right-hand bar. Note that the scaffold contrast is much milder here than the similarity contrast, and milder than in the Python edition of this unit (0.60 against 0.79 there, with a different random split): a Bemis–Murcko scaffold is a coarse grouping, two different scaffolds can still share most of their circular substructures, and which compounds land in the test fifth matters. Unit 12 replaces this post-hoc slice by a split that enforces the separation.</p>

<div class="box warn"><h4>What to do with an error analysis</h4>
<p>An error analysis is not a confession, it is a specification. This one says: report the model as a ranking tool with an expected error near 0.68 log units inside well-populated regions of chemical space and near 0.80 at its edge; flag any prediction whose nearest training neighbour is below Tanimoto 0.6; re-measure the worst twelve compounds before calling them activity cliffs, since ten of them rest on a single unreplicated measurement; and do not use the model for absolute potency at the extremes, where the regression-to-the-mean bias reaches 0.7–0.9 log units.</p></div>

## 8. The leakage catalogue

**Leakage** is the use, during training or model selection, of information that would not be available at prediction time — or the use of the evaluation data to make any decision at all. Its signature is a model that performs beautifully in the paper and poorly in the laboratory. It has five standard forms<sup class="c"><a href="#r2">[2]</a></sup><sup class="c"><a href="#r3">[3]</a></sup>, to which chemistry adds its own biases in how benchmark sets are assembled<sup class="c"><a href="#r29">[29]</a></sup>; tuning on the same data used for assessment is a sixth, measured in Unit 12<sup class="c"><a href="#r4">[4]</a></sup>.

1. **Preprocessing fitted on all the data.** Scaling, imputation, encoding or PCA computed before the split. Usually a small effect, always avoidable by keeping the step inside `caret::train`, a recipe or a closure.
2. **Supervised feature selection before the split.** Choosing features by their correlation with the target on the whole dataset. This is the dangerous one, because the selection itself is a model fitted to the test labels. Ambroise and McLachlan showed on microarray data that it can turn pure noise into an apparently excellent classifier<sup class="c"><a href="#r18">[18]</a></sup>.
3. **Duplicates across the split.** The same compound (or patient, or near-identical analogue) present in both training and test. Our curation collapsed exact duplicates by InChIKey, but analogue series are a softer version of the same problem — the subject of the scaffold split.
4. **Target-derived features.** A feature computed from the target, or from something downstream of it. In our table `pActivity_min`, `pActivity_max` and `pActivity_sd` are of this kind: innocent-looking provenance columns that are functions of the answer.
5. **Temporal leakage.** Training on data from after the prediction time — publications, assay campaigns or clinical visits that would not have existed yet. Sheridan's time-split cross-validation<sup class="c"><a href="#r5">[5]</a></sup> is the remedy; Unit 12 implements it.

### 8.1 Measuring the optimism of leaked feature selection

The experiment: select the 50 fingerprint bits most correlated with pActivity, then fit a ridge regression — once with the selection performed on the whole dataset before cross-validation (the naive version), once with the selection repeated inside each training fold (the correct version). Everything else is identical. We repeat it at four sample sizes, because the severity of this leak depends on how many rows there are per candidate feature.



In [ ]:
## The canonical leakage experiment: supervised feature selection inside vs outside the cross-validation.
select_k_best <- function(X, yy, k = 50) {
  cc <- abs(as.numeric(cor(as.matrix(X), yy)))   ## |Pearson r| ranks bits as the F statistic does
  cc[is.na(cc)] <- 0                             ## bits that are constant in this subsample
  order(cc, decreasing = TRUE)[seq_len(k)]
}
leak <- rbindlist(lapply(c(150, 400, 1000, length(y)), function(n) {
  set.seed(0); sub <- sample(seq_along(y), n)
  Xs <- as.matrix(X_fp[sub, ]); ys <- y[sub]
  set.seed(0); fo <- caret::createFolds(ys, k = 5)
  keep_all <- select_k_best(Xs, ys, 50)          ## fitted on ALL n rows, test labels included
  naive <- mean(sapply(fo, function(te) {
    tr <- setdiff(seq_along(ys), te)
    r2(ys[te], f_ridge_fixed(Xs[tr, keep_all, drop = FALSE], ys[tr], Xs[te, keep_all, drop = FALSE]))
  }))
  good <- mean(sapply(fo, function(te) {
    tr <- setdiff(seq_along(ys), te)
    kk <- select_k_best(Xs[tr, , drop = FALSE], ys[tr], 50)   ## refitted inside the fold
    r2(ys[te], f_ridge_fixed(Xs[tr, kk, drop = FALSE], ys[tr], Xs[te, kk, drop = FALSE]))
  }))
  data.table(n_compounds = n, naive_R2 = round(naive, 3), correct_R2 = round(good, 3),
             optimism = round(naive - good, 3))
}))
knitr::kable(leak, caption = "Optimism of selecting 50 of 2,048 bits outside the cross-validation.")



<p class="caption"><b>Table (Section 8.1).</b> The optimism of selecting features outside the cross-validation, as a function of sample size. With 150 compounds and 2,048 candidate bits the naive protocol reports \(R^2\) = +0.140 while the honest protocol reports \(R^2\) = −0.100 — the "model" is in fact worse than predicting the mean, and the naive number is pure selection bias worth 0.240 \(R^2\) units. The optimism falls to 0.105 at n = 400, 0.044 at n = 1,000 and 0.010 at the full 2,764 compounds. The lesson is not "leakage is small on big datasets"; it is that <em>the same wrong code produces a catastrophic lie on a small dataset and a mild one on a large dataset</em>, and small datasets are exactly what a thesis project has. (The Python edition reports a larger optimism at n = 150, about 0.50 \(R^2\) units, because its ridge uses a fixed penalty on standardised features while ours uses \(\lambda = 0.05\) on the raw bits; the direction and the shape of the curve are the same.)</p>



In [ ]:
## The optimism curve, and where a thesis-sized dataset sits on it.
lp <- melt(leak, id.vars = "n_compounds", measure.vars = c("naive_R2", "correct_R2"))
print(ggplot(lp, aes(n_compounds, value, colour = variable)) +
  geom_hline(yintercept = 0, linewidth = 0.3) +
  geom_line() + geom_point(size = 1.6) +
  scale_x_log10() +
  scale_colour_manual(values = c(naive_R2 = unname(PAL["orange"]), correct_R2 = unname(PAL["blue"])),
                      labels = c("selection outside the CV", "selection inside the CV"), name = NULL) +
  labs(x = "compounds in the experiment (log scale)", y = expression(cross-validated~R^2),
       title = "The two protocols converge only when n is large") +
  theme(legend.position = "top"))



<p class="caption"><b>Figure 9.</b> The same four experiments as a figure. The orange line (selection fitted on all rows) is almost flat in \(n\): it reports \(R^2\) between 0.14 and 0.33 whatever the sample size, because the selection can always find 50 bits that happen to correlate with the 150 or 2,764 labels it was shown. The blue line (selection refitted in each fold) starts <em>below zero</em> at n = 150 and rises to 0.312 — it tells the truth, and the truth is that 150 compounds do not support a 50-feature model. The vertical distance between the lines is the optimism; it closes only because the honest estimate improves, never because the naive one becomes more modest.</p>



In [ ]:
## The same experiment on pure noise, where the honest answer is known to be zero.
set.seed(0)
X_noise <- matrix(rnorm(100 * 2000), nrow = 100); y_noise <- rnorm(100)
set.seed(0); fo <- caret::createFolds(y_noise, k = 5)
keep_all <- select_k_best(X_noise, y_noise, 20)
naive_noise <- mean(sapply(fo, function(te) {
  tr <- setdiff(1:100, te)
  r2(y_noise[te], f_ridge_fixed(X_noise[tr, keep_all, drop = FALSE], y_noise[tr], X_noise[te, keep_all, drop = FALSE]))
}))
good_noise <- mean(sapply(fo, function(te) {
  tr <- setdiff(1:100, te); kk <- select_k_best(X_noise[tr, ], y_noise[tr], 20)
  r2(y_noise[te], f_ridge_fixed(X_noise[tr, kk, drop = FALSE], y_noise[tr], X_noise[te, kk, drop = FALSE]))
}))
bars <- data.table(experiment = rep(c("100 x 2000 pure noise", "150 real compounds"), each = 2),
                   protocol = rep(c("naive", "correct"), 2),
                   R2 = c(naive_noise, good_noise, leak$naive_R2[1], leak$correct_R2[1]))
print(ggplot(bars, aes(protocol, R2, fill = protocol)) +
  geom_col(width = 0.5) + geom_hline(yintercept = 0, linewidth = 0.4) +
  facet_wrap(~experiment) +
  scale_fill_manual(values = c(naive = unname(PAL["orange"]), correct = unname(PAL["blue"])), guide = "none") +
  labs(x = NULL, y = expression(cross-validated~R^2),
       title = "Selecting features before the split invents signal"))
cat(sprintf("pure noise  : naive R2 = %+.3f | correct R2 = %+.3f\n", naive_noise, good_noise))



<p class="caption"><b>Figure 10.</b> Leakage demonstrated where the truth is known. Left panel: a matrix of 100 rows and 2,000 columns of Gaussian noise with a Gaussian target — there is nothing to learn, and the honest protocol correctly reports \(R^2\) = −0.227, while selecting the 20 "best" columns on the full data first reports \(R^2\) = +0.563, a confident model of nothing. Right panel: the n = 150 row of the previous table, showing the same mechanism on real chemistry. Any published model whose preprocessing touched the test labels is reporting a number from the orange bars.</p>

### 8.2 Target-derived features

The fourth leak needs no cross-validation subtlety to be fatal. The curated table carries `pActivity_min`, `pActivity_max` and `pActivity_sd`, which describe the replicate measurements from which the target was computed. They are legitimate provenance columns and illegitimate features.



In [ ]:
## What happens when a provenance column is left in the feature matrix by accident.
X_leaky <- cbind(X_desc, as.matrix(reg[, .(pActivity_min, pActivity_max, pActivity_sd)]))
X_leaky[is.na(X_leaky)] <- 0
set.seed(0); fo_all <- caret::createFolds(y, k = 5)
ev <- function(X) {
  m <- sapply(fo_all, function(te) {
    tr <- setdiff(seq_along(y), te)
    p  <- f_ridge_fixed(X[tr, , drop = FALSE], y[tr], X[te, , drop = FALSE])
    c(R2 = r2(y[te], p), RMSE = rmse(y[te], p))
  })
  rowMeans(m)
}
a <- ev(X_desc); b <- ev(X_leaky)
cat(sprintf("descriptors only                 : R2 = %.3f | RMSE = %.3f\n", a[["R2"]], a[["RMSE"]]))
cat(sprintf("descriptors + provenance columns : R2 = %.3f | RMSE = %.3f\n", b[["R2"]], b[["RMSE"]]))
cat("for compounds with a single measurement, pActivity_min == pActivity:",
    all(abs(reg[n_measurements == 1, pActivity_min - pActivity]) < 1e-9), "\n")



\(R^2\) jumps from 0.116 to 0.996 and RMSE collapses from 1.042 to 0.071 log units. No cross-validation scheme protects against this: the leak is not in the protocol but in the matrix, since for compounds measured once `pActivity_min` *is* the target, as the printed `TRUE` confirms. A result this good is always a bug, and the habit that catches it is to read the feature columns aloud and ask of each: *would I know this value for a molecule never tested?*

<div class="box warn"><h4>The leakage checklist</h4>
<p>Before reporting any model: (1) is every learned preprocessing step fitted inside the resample? (2) was any feature chosen, any threshold set, any hyperparameter tuned using the test data? (3) can the same entity (compound, patient, trajectory) appear on both sides of the split? (4) could any feature be computed only <em>after</em> the target was known? (5) does the model still fail on shuffled labels? (6) is the reported score better than the field's typical result for this kind of data — and if so, why?</p></div>

## 9. The model card

A model that leaves your laptop needs a short document stating what it is, what it was trained on, how it was evaluated and where it should not be used — a model card<sup class="c"><a href="#r17">[17]</a></sup>, whose biomedical equivalents are the DOME recommendations<sup class="c"><a href="#r15">[15]</a></sup> and TRIPOD+AI<sup class="c"><a href="#r16">[16]</a></sup> and whose QSAR version is the OECD validation principles<sup class="c"><a href="#r24">[24]</a></sup> elaborated by Tropsha<sup class="c"><a href="#r7">[7]</a></sup>. The next chunk writes the card from the objects in memory, so it cannot drift from the model it describes.



In [ ]:
## A model card generated from the run itself, not from memory.
card <- c(
  "model"            = "ranger(num.trees = 300, seed = 1) on frozen Morgan(r = 2, 2048 bit) fingerprints",
  "task"             = "regression of pActivity (-log10 M) for ROCK1 (CHEMBL3231) inhibitors",
  "unit of prediction" = "one standardised parent structure (InChIKey)",
  "training data"    = sprintf("%d compounds from the curated ChEMBL snapshot (exact measurements only)", length(idx_tr)),
  "excluded"         = sprintf("%d censored compounds (all measurements '>' or '<')", sum(df$censored)),
  "split"            = "random 80/20 by compound, set.seed(0); inner 5-fold CV for all comparisons",
  "test performance" = sprintf("RMSE %.3f, MAE %.3f, R2 %.3f, Spearman %.3f on %d held-out compounds",
                               rmse(y[idx_te], pred_te), mae(y[idx_te], pred_te), r2(y[idx_te], pred_te),
                               cor(y[idx_te], pred_te, method = "spearman"), length(idx_te)),
  "baseline"         = sprintf("predicting the training mean gives RMSE %.3f", sd(ytr)),
  "performance by applicability" = sprintf("RMSE %.2f in the most-similar quartile vs %.2f in the least-similar; %.2f for known scaffolds vs %.2f for unseen ones",
                               byq$RMSE[4], byq$RMSE[1],
                               sqrt(mean(err[new_scaffold == FALSE, residual^2])),
                               sqrt(mean(err[new_scaffold == TRUE, residual^2]))),
  "known limitations" = sprintf("regresses to the mean at the extremes (%+.2f below pActivity 5.5, %+.2f above 7.5); the largest errors concentrate on single-measurement compounds and on activity cliffs",
                               mean(err[y < 5.5, residual]), mean(err[y > 7.5, residual])),
  "intended use"     = "ranking of analogues inside known ROCK1 chemotypes; NOT absolute potency prediction",
  "sanity checks"    = sprintf("shuffled-label R2 = %+.3f; no preprocessing fitted outside a fold", shuf[["R2"]]))
for (k in names(card)) cat(sprintf("%-29s: %s\n", k, card[[k]]))



Everything in the card is a number this document produced, and every entry answers a question a referee asks. The two entries that distinguish a careful card from a decorative one are "performance by applicability" — performance split by whether the compound resembles the training data — and "intended use", which is where you write down the thing the model is *not* for. In a real project this list is written to `model_card.md` next to the serialised model (Unit 23 turns it into a Methods section, Unit 24 into a tested package).

<div class="box bridge"><h4>Bridge — where this unit sits</h4>
<p>The neighbouring units supply what this one simplified. <a href="../U12_Validation/index.html">Unit 12 (R)</a> replaces the random split by the full split taxonomy and puts confidence intervals on every number above; <a href="../U13_Features_Dimensionality/index.html">Unit 13 (R)</a> does feature selection properly, with stability across folds; <a href="../U14_Ensembles_Tuning/index.html">Unit 14 (R)</a> tunes the forest and its boosted cousins under a budget. The <a href="../index.html">R-edition index</a> lists the remaining units — Unit 9 explains the ridge baseline, Unit 17 turns "similarity below 0.6" into a calibrated applicability domain, Unit 23 turns the model card into a Methods section — and each has a Python counterpart under <a href="../../Python/index.html">Material/</a>; the Python version of this very unit is <a href="../../Python/U11_Supervised_Workflow/index.html">here</a>. The algorithms themselves are the subject of the artificial-intelligence course, whose R edition covers model evaluation in <a href="../../../BGM411/R/S1_Model_Evaluation/index.html">S1</a> and trees and forests in <a href="../../../BGM411/R/U4_Trees_Random_Forest/index.html">U4</a>; the chemistry of the representation is <a href="../../../Cheminformatics/U5_Descriptors_Fingerprints/index.html">Unit 5 of the cheminformatics course</a> and the curation behind the table is <a href="../../../Cheminformatics/U7_Data_Curation/index.html">its Unit 7</a>. For the structural side of ROCK1 see <a href="../../../BGM324/03_Molecular_Docking/index.html">BGM324 Unit 3</a>.</p></div>

## 10. Summary

You should now be able to explain, without notes:

- the seven decisions of a supervised project, and why only one is about the algorithm;
- why the unit of prediction must be the entity you will be asked to predict (compound, not measurement; patient, not tissue site);
- when a question is a regression, a classification at a stated threshold, or a ranking problem, with the metric for each;
- what `caret::train(preProcess = ...)`, a `recipes` recipe and a fit/transform closure guarantee, and which featurisation steps are safe outside them;
- why a score is meaningless without a baseline, and which three baselines to compute;
- how to read a learning curve and decide between more data and more capacity;
- how to turn an error analysis into a statement about the applicability of the model;
- the five leakage sources, the optimism each produces, and the checklist that catches them;
- what belongs in a model card.

The single number to take away: on this dataset, selecting 50 of 2,048 features outside the cross-validation inflates \(R^2\) by 0.240 at n = 150 and by 0.010 at n = 2,764 — and on pure noise it manufactures an \(R^2\) of +0.563 out of nothing.

## 11. Self-assessment

<details><summary>1. A colleague reports an \(R^2\) of 0.72 for a QSAR model. What are the first three questions you ask?</summary>Computed on what — cross-validation or a held-out set, under which split? Against which baseline? And what is the unit of prediction, compounds or measurements? Without those three the number is not interpretable.</details>

<details><summary>2. Why is a 1-nearest-neighbour model on Tanimoto similarity a more meaningful baseline than the mean?</summary>Because it is what a chemist does without a computer: find the closest known analogue and assume similar potency. Beating 1-NN shows the model has learned something beyond "similar molecules have similar activity" — here 1-NN reaches \(R^2\) = 0.243 against the forest's 0.543.</details>

<details><summary>3. You standardise the full data matrix and then split. How bad is it, and how would you find out?</summary>Usually mild, because scaling is unsupervised and a mean from 80% of the rows differs little from one over 100% — 382.03 against 381.59 g/mol here. You find out by running both protocols, as in Section 8.1. Avoid it anyway: the same habit applied to a supervised step is catastrophic.</details>

<details><summary>4. Which of these can safely be computed before the split: a Morgan fingerprint, a centring and scaling step, a PCA rotation, a molecular weight, a filter keeping bits present in more than 1% of compounds?</summary>The fingerprint and the molecular weight — row-wise functions of one molecule. The scaler, the PCA and the frequency filter learn from the set as a whole and must be fitted inside the training fold; the filter in Section 3.4 kept a different number of bits (719 to 736) in each fold, which is the visible proof.</details>

<details><summary>5. Training RMSE 0.43, validation RMSE 0.74, validation curve still falling. What next?</summary>Acquire more compounds: the curve has not plateaued. Reducing variance would also narrow the gap, but the slope says data is the cheapest remaining gain. Contrast Figure 6, where the curves converged at 1.04 and extra data is worthless.</details>

<details><summary>6. Your test \(R^2\) is slightly different from your cross-validated \(R^2\). Is something wrong?</summary>No. A single test set of 553 compounds is one random draw; here the test RMSE is 0.738 against a cross-validated 0.742, a difference of 0.004 inside the fold-to-fold standard deviation of 0.027. The response is a confidence interval, not a comparison of two point estimates — Unit 12.</details>

<details><summary>7. Why does the model predict weak compounds as too potent and potent compounds as too weak?</summary>Regression to the mean: minimising squared error against a bell-shaped target shrinks predictions towards the centre, most strongly where data are sparse. Absolute potency at the extremes is therefore unreliable while the ranking stays informative — Spearman \(\rho\) = 0.774 with \(R^2\) = 0.585.</details>

<details><summary>8. A feature called <code>n_assays</code> improves your model substantially. Should you keep it?</summary>No. It is known only because the compound was tested, so it cannot be computed for a new molecule, and it correlates with potency because interesting compounds get re-tested. Same family as <code>pActivity_min</code>, which pushed \(R^2\) to 0.996.</details>

<details><summary>9. What does a negative cross-validated \(R^2\) mean?</summary>That the model predicts held-out data worse than the training mean. It is the correct result for a model fitted to noise (−0.227 in Section 8.1) and common on small datasets; \(R^2\) is not bounded below by zero out of sample.</details>

<details><summary>10. How would you convert this regression into the ranking problem a screening campaign needs?</summary>Keep the chain but evaluate with ranking metrics on the top slice: enrichment factor at 1%, BEDROC, Spearman \(\rho\) among the predicted top 100 (Unit 19). The change is in decisions 1, 6 and 7 — question, metric, error analysis — not in the algorithm.</details>

<details><summary>11. Your chain gives \(R^2\) = 0.93 where the literature reports 0.6. What do you do before celebrating?</summary>Assume leakage and run the checklist: which feature columns could not be known before the target; duplicates straddling folds; selection or tuning that saw the validation folds; shuffled labels. Results far above the field's norm are usually a bug.</details>

<details><summary>12. Why does the model card record performance separately for similar and dissimilar compounds?</summary>Because the user is almost always asking about the dissimilar case. RMSE is 0.680 in the most-similar quartile and 0.802 in the least-similar; quoting only the pooled 0.738 overstates performance for the prospective use.</details>

## Glossary

<dl class="gloss">
<dt>Baseline</dt><dd>A deliberately simple predictor (the mean, the majority class, the nearest neighbour, a regularised linear model) whose score any real model must beat before it is worth discussing.</dd>
<dt>Censored measurement</dt><dd>A record stating only that a value lies above or below a bound ("IC50 > 10 µM"); usable as an inequality, not as a number.</dd>
<dt>Closure (R)</dt><dd>A function that returns another function together with the data it captured; the lightest way to express "fit once, apply many times" without a class system.</dd>
<dt>Descriptor</dt><dd>A single interpretable number computed from a structure (weight, lipophilicity, polar surface area, ring count...).</dd>
<dt>Error analysis</dt><dd>Study of which rows a model predicts worst and why; usually a ranked residual table annotated with domain information.</dd>
<dt>Learning curve</dt><dd>Training and validation error against training-set size; diagnoses underfitting (converged, high) versus overfitting (wide, persistent gap).</dd>
<dt>Leakage</dt><dd>Use at training or selection time of information unavailable at prediction time, including any use of the evaluation data to make a decision.</dd>
<dt>MAE / RMSE</dt><dd>Mean absolute error and root mean squared error; RMSE squares residuals and is dominated by the worst cases.</dd>
<dt>Model card</dt><dd>A short document stating what a model is, what it was trained and evaluated on, how it performs by subgroup, and where it must not be used.</dd>
<dt>Morgan fingerprint (ECFP)</dt><dd>A binary vector (here 2,048 bits) recording which circular atom environments a molecule contains; loaded in this edition from a frozen sparse file generated with RDKit.</dd>
<dt>pActivity</dt><dd>\(-\log_{10}\) of an activity concentration in molar; larger means more potent, one unit is tenfold.</dd>
<dt>Recipe (<code>recipes</code>)</dt><dd>A declarative list of preprocessing steps, estimated with <code>prep()</code> on training data and applied with <code>bake()</code>; the R counterpart of a column transformer.</dd>
<dt>R²</dt><dd>One minus residual over total sum of squares; relative to the variance of the evaluation set, and negative out of sample for a model worse than the mean.</dd>
<dt>Regression to the mean</dt><dd>The shrinkage of squared-error predictions towards the centre of the target distribution, seen as a tilt against the identity line.</dd>
<dt>Scaffold (Bemis–Murcko)</dt><dd>Ring systems plus linkers, side chains removed; a computable proxy for "chemical series".</dd>
<dt>Tanimoto (Jaccard) similarity</dt><dd>Shared bits over union of bits between two fingerprints; the standard similarity for binary substructure vectors.</dd>
<dt>Unit of prediction</dt><dd>The entity one row represents and the model is asked to predict; must be the entity the question is about.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Kuhn M. Building predictive models in R using the caret package. <em>J Stat Softw</em>. 2008;28(5):1-26. <a href="https://doi.org/10.18637/jss.v028.i05">doi:10.18637/jss.v028.i05</a></li>
<li id="r2">Kaufman S, Rosset S, Perlich C, Stitelman O. Leakage in data mining: formulation, detection, and avoidance. <em>ACM Trans Knowl Discov Data</em>. 2012;6(4):15. <a href="https://doi.org/10.1145/2382577.2382579">doi:10.1145/2382577.2382579</a></li>
<li id="r3">Kapoor S, Narayanan A. Leakage and the reproducibility crisis in machine-learning-based science. <em>Patterns</em>. 2023;4(9):100804. <a href="https://doi.org/10.1016/j.patter.2023.100804">doi:10.1016/j.patter.2023.100804</a></li>
<li id="r4">Varma S, Simon R. Bias in error estimation when using cross-validation for model selection. <em>BMC Bioinformatics</em>. 2006;7:91. <a href="https://doi.org/10.1186/1471-2105-7-91">doi:10.1186/1471-2105-7-91</a></li>
<li id="r5">Sheridan RP. Time-split cross-validation as a method for estimating the goodness of prospective prediction. <em>J Chem Inf Model</em>. 2013;53(4):783-790. <a href="https://doi.org/10.1021/ci400084k">doi:10.1021/ci400084k</a></li>
<li id="r6">Whalen S, Schreiber J, Noble WS, Pollard KS. Navigating the pitfalls of applying machine learning in genomics. <em>Nat Rev Genet</em>. 2022;23(3):169-181. <a href="https://doi.org/10.1038/s41576-021-00434-9">doi:10.1038/s41576-021-00434-9</a></li>
<li id="r7">Tropsha A. Best practices for QSAR model development, validation, and exploitation. <em>Mol Inform</em>. 2010;29(6-7):476-488. <a href="https://doi.org/10.1002/minf.201000061">doi:10.1002/minf.201000061</a></li>
<li id="r8">Golbraikh A, Tropsha A. Beware of q2! <em>J Mol Graph Model</em>. 2002;20(4):269-276. <a href="https://doi.org/10.1016/S1093-3263(01)00123-1">doi:10.1016/S1093-3263(01)00123-1</a></li>
<li id="r9">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL Database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r10">Rogers D, Hahn M. Extended-connectivity fingerprints. <em>J Chem Inf Model</em>. 2010;50(5):742-754. <a href="https://doi.org/10.1021/ci100050t">doi:10.1021/ci100050t</a></li>
<li id="r11">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
<li id="r12">Bemis GW, Murcko MA. The properties of known drugs. 1. Molecular frameworks. <em>J Med Chem</em>. 1996;39(15):2887-2893. <a href="https://doi.org/10.1021/jm9602928">doi:10.1021/jm9602928</a></li>
<li id="r13">Kalliokoski T, Kramer C, Vulpetti A, Gedeck P. Comparability of mixed IC50 data - a statistical analysis. <em>PLoS ONE</em>. 2013;8(4):e61007. <a href="https://doi.org/10.1371/journal.pone.0061007">doi:10.1371/journal.pone.0061007</a></li>
<li id="r14">Landrum GA, Riniker S. Combining IC50 or Ki values from different sources is a source of significant noise. <em>J Chem Inf Model</em>. 2024;64(5):1560-1567. <a href="https://doi.org/10.1021/acs.jcim.4c00049">doi:10.1021/acs.jcim.4c00049</a></li>
<li id="r15">Walsh I, Fishman D, Garcia-Gasulla D, et al. DOME: recommendations for supervised machine learning validation in biology. <em>Nat Methods</em>. 2021;18(10):1122-1127. <a href="https://doi.org/10.1038/s41592-021-01205-4">doi:10.1038/s41592-021-01205-4</a></li>
<li id="r16">Collins GS, Moons KGM, Dhiman P, et al. TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. <em>BMJ</em>. 2024;385:e078378. <a href="https://doi.org/10.1136/bmj-2023-078378">doi:10.1136/bmj-2023-078378</a></li>
<li id="r17">Mitchell M, Wu S, Zaldivar A, et al. Model cards for model reporting. In: <em>Proc. Conf. on Fairness, Accountability, and Transparency (FAT* '19)</em>. ACM; 2019:220-229. <a href="https://doi.org/10.1145/3287560.3287596">doi:10.1145/3287560.3287596</a></li>
<li id="r18">Ambroise C, McLachlan GJ. Selection bias in gene extraction on the basis of microarray gene-expression data. <em>Proc Natl Acad Sci USA</em>. 2002;99(10):6562-6566. <a href="https://doi.org/10.1073/pnas.102102699">doi:10.1073/pnas.102102699</a></li>
<li id="r19">Wright MN, Ziegler A. ranger: a fast implementation of random forests for high dimensional data in C++ and R. <em>J Stat Softw</em>. 2017;77(1):1-17. <a href="https://doi.org/10.18637/jss.v077.i01">doi:10.18637/jss.v077.i01</a></li>
<li id="r20">Svetnik V, Liaw A, Tong C, Culberson JC, Sheridan RP, Feuston BP. Random forest: a classification and regression tool for compound classification and QSAR modeling. <em>J Chem Inf Comput Sci</em>. 2003;43(6):1947-1958. <a href="https://doi.org/10.1021/ci034160g">doi:10.1021/ci034160g</a></li>
<li id="r21">Friedman J, Hastie T, Tibshirani R. Regularization paths for generalized linear models via coordinate descent. <em>J Stat Softw</em>. 2010;33(1):1-22. <a href="https://doi.org/10.18637/jss.v033.i01">doi:10.18637/jss.v033.i01</a></li>
<li id="r22">Mathea M, Klingspohn W, Baumann K. Chemoinformatic classification methods and their applicability domain. <em>Mol Inform</em>. 2016;35(5):160-180. <a href="https://doi.org/10.1002/minf.201501019">doi:10.1002/minf.201501019</a></li>
<li id="r23">Hawkins DM. The problem of overfitting. <em>J Chem Inf Comput Sci</em>. 2004;44(1):1-12. <a href="https://doi.org/10.1021/ci0342472">doi:10.1021/ci0342472</a></li>
<li id="r24">Gramatica P. Principles of QSAR models validation: internal and external. <em>QSAR Comb Sci</em>. 2007;26(5):694-701. <a href="https://doi.org/10.1002/qsar.200610151">doi:10.1002/qsar.200610151</a></li>
<li id="r25">Bickerton GR, Paolini GV, Besnard J, Muresan S, Hopkins AL. Quantifying the chemical beauty of drugs. <em>Nat Chem</em>. 2012;4:90-98. <a href="https://doi.org/10.1038/nchem.1243">doi:10.1038/nchem.1243</a></li>
<li id="r26">Breiman L. Random forests. <em>Mach Learn</em>. 2001;45:5-32. <a href="https://doi.org/10.1023/A:1010933404324">doi:10.1023/A:1010933404324</a></li>
<li id="r27">Todeschini R, Consonni V. <em>Molecular Descriptors for Chemoinformatics</em>. 2nd ed. Wiley-VCH; 2009. <a href="https://doi.org/10.1002/9783527628766">doi:10.1002/9783527628766</a></li>
<li id="r28">Stumpfe D, Bajorath J. Exploring activity cliffs in medicinal chemistry. <em>J Med Chem</em>. 2012;55(7):2932-2942. <a href="https://doi.org/10.1021/jm201706b">doi:10.1021/jm201706b</a></li>
<li id="r29">Sieg J, Flachsenberg F, Rarey M. In need of bias control: evaluating chemical data for machine learning in structure-based virtual screening. <em>J Chem Inf Model</em>. 2019;59(3):947-961. <a href="https://doi.org/10.1021/acs.jcim.8b00712">doi:10.1021/acs.jcim.8b00712</a></li>
<li id="r30">Hoerl AE, Kennard RW. Ridge regression: biased estimation for nonorthogonal problems. <em>Technometrics</em>. 1970;12(1):55-67. <a href="https://doi.org/10.1080/00401706.1970.10488634">doi:10.1080/00401706.1970.10488634</a></li>
<li id="r31">Kuhn M, Johnson K. <em>Applied Predictive Modeling</em>. Springer; 2013. <a href="https://doi.org/10.1007/978-1-4614-6849-3">doi:10.1007/978-1-4614-6849-3</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021 (free PDF). — Chapters 2 and 5 for bias/variance and resampling, in R.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009 (free PDF). — Section 7.10.2 is the formal treatment of "the wrong way to do cross-validation".</li>
<li>Kuhn M, Silge J. <em>Tidy Modeling with R</em>. O'Reilly; 2022 (free online). — the recipes/workflows idiom used in Section 3.3.</li>
<li>Wickham H, Çetinkaya-Rundel M, Grolemund G. <em>R for Data Science</em>. 2nd ed. O'Reilly; 2023 (free online). — the data-manipulation and plotting background assumed here.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li><code>caret</code> documentation, "Pre-Processing" and "Model Training and Tuning": <a href="https://topepo.github.io/caret/">topepo.github.io/caret</a>.</li>
<li>scikit-learn user guide, "Common pitfalls and recommended practices" — language-independent and worth reading next to this unit: <a href="https://scikit-learn.org/stable/common_pitfalls.html">scikit-learn.org/stable/common_pitfalls.html</a>.</li>
<li>RDKit documentation (the generator that produced the frozen fingerprints): <a href="https://www.rdkit.org/docs/">rdkit.org/docs</a>.</li>
<li>Practical Cheminformatics (P. Walters), especially the posts on baselines and on evaluating QSAR models: <a href="https://patwalters.github.io/practicalcheminformatics/">patwalters.github.io/practicalcheminformatics</a>.</li>
<li><em>The Turing Way</em> — guide to reproducible research: <a href="https://the-turing-way.netlify.app/">the-turing-way.netlify.app</a>.</li>
</ul>
